# 1. Imports

In [1]:
import os, gc, json, warnings
import numpy as np
import pandas as pd
import torch

from datasets import Dataset, DatasetDict
from transformers import (
    AutoTokenizer,
    AutoModelForSeq2SeqLM,
    DataCollatorForSeq2Seq,
    Seq2SeqTrainer,
    Seq2SeqTrainingArguments,
    set_seed,
)

warnings.filterwarnings("ignore")
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", DEVICE)
if DEVICE == "cuda":
    print("GPU:", torch.cuda.get_device_name(0))

Device: cuda
GPU: Tesla T4


# 2. Configurations

In [2]:
from dataclasses import dataclass

@dataclass
class Config:
    # ----- Paths (EDIT THESE) -----------------------------------------
    train_csv: str = "/kaggle/input/datasets/mishbhaul/train-csv/mixed_train_70k.csv"
    output_dir: str = "/kaggle/working/helsinki-ar-en"
    final_weights_dir: str = "/kaggle/working/helsinki-ar-en-final"
    src_col: str = "ar"   # CSV column with Arabic text
    tgt_col: str = "en"    # CSV column with English text

    # ----- Model -------------------------------------------------------
    # model_name: str = "Helsinki-NLP/opus-mt-en-ar"
    model_name: str = "Helsinki-NLP/opus-mt-tc-big-ar-en"
    # ----- Data --------------------------------------------------------
    max_samples: int = 70_000
    val_size: float = 0.05
    test_size: float = 0.05
    seed: int = 42
    max_src_len: int = 128
    max_tgt_len: int = 128

    # ----- Training ----------------------------------------------------
    num_train_epochs: int = 3
    per_device_train_batch_size: int = 16
    per_device_eval_batch_size: int = 16
    gradient_accumulation_steps: int = 2
    learning_rate: float = 5e-5
    weight_decay: float = 0.01
    warmup_ratio: float = 0.1
    label_smoothing_factor: float = 0.1
    fp16: bool = True              # auto-disabled below if no GPU
    logging_steps: int = 100
    save_total_limit: int = 2
    generation_num_beams: int = 4

    # ----- Metrics -----------------------------------------------------
    compute_comet: bool = True
    comet_model: str = "Unbabel/wmt22-comet-da"
    comet_max_eval_samples: int = 500  # subsample COMET for speed

cfg = Config()
cfg.fp16 = cfg.fp16 and torch.cuda.is_available()
set_seed(cfg.seed)
os.makedirs(cfg.output_dir, exist_ok=True)
os.makedirs(cfg.final_weights_dir, exist_ok=True)
print(json.dumps(cfg.__dict__, indent=2, ensure_ascii=False))

{
  "train_csv": "/kaggle/input/datasets/mishbhaul/train-csv/mixed_train_70k.csv",
  "output_dir": "/kaggle/working/helsinki-ar-en",
  "final_weights_dir": "/kaggle/working/helsinki-ar-en-final",
  "src_col": "ar",
  "tgt_col": "en",
  "model_name": "Helsinki-NLP/opus-mt-tc-big-ar-en",
  "max_samples": 70000,
  "val_size": 0.05,
  "test_size": 0.05,
  "seed": 42,
  "max_src_len": 128,
  "max_tgt_len": 128,
  "num_train_epochs": 3,
  "per_device_train_batch_size": 16,
  "per_device_eval_batch_size": 16,
  "gradient_accumulation_steps": 2,
  "learning_rate": 5e-05,
  "weight_decay": 0.01,
  "warmup_ratio": 0.1,
  "label_smoothing_factor": 0.1,
  "fp16": true,
  "logging_steps": 100,
  "save_total_limit": 2,
  "generation_num_beams": 4,
  "compute_comet": true,
  "comet_model": "Unbabel/wmt22-comet-da",
  "comet_max_eval_samples": 500
}


# 3 · Data — Load, Clean, Split, Tokenize

In [3]:
def load_raw_dataframe(cfg):
    '''Read CSV, validate columns, clean, cap at max_samples.'''
    df = pd.read_csv(cfg.train_csv)
    for col in (cfg.src_col, cfg.tgt_col):
        if col not in df.columns:
            raise ValueError(
                f"Column '{col}' not in CSV. Available: {list(df.columns)}"
            )
    df = df[[cfg.src_col, cfg.tgt_col]].dropna().copy()
    df[cfg.src_col] = df[cfg.src_col].astype(str).str.strip()
    df[cfg.tgt_col] = df[cfg.tgt_col].astype(str).str.strip()
    df = df[(df[cfg.src_col] != "") & (df[cfg.tgt_col] != "")]
    df = df.drop_duplicates().reset_index(drop=True)
    if len(df) > cfg.max_samples:
        df = df.sample(n=cfg.max_samples, random_state=cfg.seed)
        df = df.reset_index(drop=True)
    print(f"[data] {len(df)} clean pairs loaded")
    return df


def build_dataset_dict(cfg, df):
    '''Split into train/validation/test DatasetDict.'''
    ds = Dataset.from_pandas(df, preserve_index=False)
    s1 = ds.train_test_split(test_size=cfg.test_size, seed=cfg.seed)
    val_frac = cfg.val_size / (1.0 - cfg.test_size)
    s2 = s1["train"].train_test_split(test_size=val_frac, seed=cfg.seed)
    dsd = DatasetDict(train=s2["train"],
                      validation=s2["test"],
                      test=s1["test"])
    print(f"[data] train={len(dsd['train'])}  "
          f"val={len(dsd['validation'])}  test={len(dsd['test'])}")
    return dsd


raw_df = load_raw_dataframe(cfg)
raw_dsd = build_dataset_dict(cfg, raw_df)
raw_df.head(3)

[data] 70000 clean pairs loaded
[data] train=62999  val=3501  test=3500


,ar,en
0,بيركلي ـ هذا هو موسم مؤتمرات النقد الدولية. فف...,"In March, national leaders assembled in Nanjin..."
1,ولكن هل طلبت أميركا من الآخرين حقاً أن يستجلبو...,Had America really told others to bring in Ame...
2,لذا، لا ينبغي لنا أن نتسرع في رفض مقترحات من ي...,So we shouldn’t be too quick to dismiss the su...


# 4. Load Model

In [4]:
def load_model_and_tokenizer(cfg):
    tokenizer = AutoTokenizer.from_pretrained(cfg.model_name)
    model = AutoModelForSeq2SeqLM.from_pretrained(cfg.model_name)
    print(f"[model] loaded '{cfg.model_name}' "
          f"({model.num_parameters()/1e6:.1f}M params)")
    return model, tokenizer

model, tokenizer = load_model_and_tokenizer(cfg)

config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json:   0%|          | 0.00/337 [00:00<?, ?B/s]

source.spm:   0%|          | 0.00/915k [00:00<?, ?B/s]

target.spm:   0%|          | 0.00/804k [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

...

Loading weights:   0%|          | 0/257 [00:00<?, ?it/s]

model.safetensors:   0%|          | 0.00/603M [00:00<?, ?B/s]

The tied weights mapping and config for this model specifies to tie model.shared.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The tied weights mapping and config for this model specifies to tie model.shared.weight to model.decoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The tied weights mapping and config for this model specifies to tie model.shared.weight to model.encoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


generation_config.json:   0%|          | 0.00/301 [00:00<?, ?B/s]

[model] loaded 'Helsinki-NLP/opus-mt-tc-big-ar-en' (428.8M params)


# 5. Tokenizing

In [5]:
def make_tokenize_fn(cfg, tokenizer):
    '''Return a batched tokenization function.'''
    def tokenize(batch):
        model_inputs = tokenizer(
            batch[cfg.src_col],
            max_length=cfg.max_src_len,
            truncation=True,
        )
        labels = tokenizer(
            text_target=batch[cfg.tgt_col],
            max_length=cfg.max_tgt_len,
            truncation=True,
        )
        model_inputs["labels"] = labels["input_ids"]
        return model_inputs
    return tokenize

tokenize_fn = make_tokenize_fn(cfg, tokenizer)
tokenized = raw_dsd.map(
    tokenize_fn,
    batched=True,
    remove_columns=raw_dsd["train"].column_names,
    desc="Tokenizing",
)

data_collator = DataCollatorForSeq2Seq(
    tokenizer, model=model, padding="longest",
    label_pad_token_id=-100,
)
print("[data] tokenization complete")

Tokenizing:   0%|          | 0/62999 [00:00<?, ? examples/s]

Tokenizing:   0%|          | 0/3501 [00:00<?, ? examples/s]

Tokenizing:   0%|          | 0/3500 [00:00<?, ? examples/s]

[data] tokenization complete


In [6]:
!pip install "sacrebleu>=2.4" "evaluate>=0.4" "unbabel-comet>=2.2"

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 55.5/55.5 kB 3.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.0/61.0 kB 4.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 2.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 100.8/100.8 kB 7.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 6.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 91.0/91.0 kB 6.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 101.4/101.4 kB 8.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 25.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.0/18.0 MB 87.2 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 295.2/295.2 kB 22.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 529.7/529.7 kB 30.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 104.5 MB/s eta 0:00:0000:010:01
  Attempting uninstall

# 6. Metrics

In [7]:
import sacrebleu

# COMET is optional & heavy — load lazily and degrade gracefully.
comet_model = None
if cfg.compute_comet:
    try:
        from comet import download_model, load_from_checkpoint
        _ckpt = download_model(cfg.comet_model)
        comet_model = load_from_checkpoint(_ckpt)
        print("[metrics] COMET model ready")
    except Exception as e:
        print(f"[metrics] COMET disabled ({e})")
        cfg.compute_comet = False


def postprocess(preds, labels):
    '''Strip whitespace; sacrebleu wants refs as list-of-list.'''
    preds = [p.strip() for p in preds]
    labels = [[l.strip()] for l in labels]
    return preds, labels


# Raw source strings aligned with the validation split (for COMET).
VAL_SOURCES = raw_dsd["validation"][cfg.src_col]


def build_compute_metrics(tokenizer, sources):
    '''Closure giving Seq2SeqTrainer a compute_metrics fn.'''
    def compute_metrics(eval_preds):
        preds, labels = eval_preds
        if isinstance(preds, tuple):
            preds = preds[0]
        preds = np.where(preds != -100, preds, tokenizer.pad_token_id)
        labels = np.where(labels != -100, labels, tokenizer.pad_token_id)

        decoded_preds = tokenizer.batch_decode(preds, skip_special_tokens=True)
        decoded_labels = tokenizer.batch_decode(labels, skip_special_tokens=True)
        decoded_preds, decoded_labels = postprocess(decoded_preds, decoded_labels)

        bleu = sacrebleu.corpus_bleu(decoded_preds, list(zip(*decoded_labels)))
        chrf = sacrebleu.corpus_chrf(decoded_preds, list(zip(*decoded_labels)),
                                     word_order=2)  # ChrF++
        results = {"bleu": round(bleu.score, 4),
                   "chrf++": round(chrf.score, 4)}

        if cfg.compute_comet and comet_model is not None:
            n = min(cfg.comet_max_eval_samples, len(decoded_preds))
            comet_data = [
                {"src": sources[i],
                 "mt": decoded_preds[i],
                 "ref": decoded_labels[i][0]}
                for i in range(n)
            ]
            out = comet_model.predict(comet_data, batch_size=32,
                                      gpus=1 if torch.cuda.is_available() else 0,
                                      progress_bar=False)
            results["comet"] = round(float(out["system_score"]), 4)

        return results
    return compute_metrics

compute_metrics = build_compute_metrics(tokenizer, VAL_SOURCES)
print("[metrics] BLEU + ChrF++" + (" + COMET" if cfg.compute_comet else ""))

[metrics] COMET disabled (cannot import name 'find_pruneable_heads_and_indices' from 'transformers.pytorch_utils' (/usr/local/lib/python3.12/dist-packages/transformers/pytorch_utils.py))
[metrics] BLEU + ChrF++


# 7. Train setup

In [8]:
training_args = Seq2SeqTrainingArguments(
    output_dir=cfg.output_dir,
    eval_strategy="epoch",
    save_strategy="epoch",
    logging_steps=cfg.logging_steps,
    learning_rate=cfg.learning_rate,
    per_device_train_batch_size=cfg.per_device_train_batch_size,
    per_device_eval_batch_size=cfg.per_device_eval_batch_size,
    gradient_accumulation_steps=cfg.gradient_accumulation_steps,
    weight_decay=cfg.weight_decay,
    warmup_ratio=cfg.warmup_ratio,
    label_smoothing_factor=cfg.label_smoothing_factor,
    num_train_epochs=cfg.num_train_epochs,
    fp16=cfg.fp16,
    predict_with_generate=True,
    generation_max_length=cfg.max_tgt_len,
    generation_num_beams=cfg.generation_num_beams,
    save_total_limit=cfg.save_total_limit,
    load_best_model_at_end=True,
    metric_for_best_model="bleu",
    greater_is_better=True,
    report_to="none",
    seed=cfg.seed,
)

trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=tokenized["train"],
    eval_dataset=tokenized["validation"],
    processing_class=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)
print("[trainer] ready")

model = model.float() 

warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


[trainer] ready


# 8. Train

In [9]:
train_result = trainer.train()

trainer.log_metrics("train", train_result.metrics)
trainer.save_metrics("train", train_result.metrics)
print("\n[train] finished")

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None}.


Epoch,Training Loss,Validation Loss,Bleu,Chrf++
1,14.198680,3.338486,26.223000,48.994200
2,12.064449,3.002118,31.259200,53.817000
3,11.255621,2.935258,32.237600,54.774100


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

***** train metrics *****
  epoch                    =        3.0
  total_flos               = 31903923GF
  train_loss               =    14.5065
  train_runtime            = 2:48:45.70
  train_samples_per_second =     18.665
  train_steps_per_second   =      0.292

[train] finished


# 9. Final eval

In [10]:
# Rebuild compute_metrics so COMET uses the *test* sources, not val sources.
compute_metrics_test = build_compute_metrics(tokenizer, raw_dsd["test"][cfg.src_col])
trainer.compute_metrics = compute_metrics_test

test_metrics = trainer.evaluate(
    eval_dataset=tokenized["test"],
    metric_key_prefix="test",
)
trainer.log_metrics("test", test_metrics)
trainer.save_metrics("test", test_metrics)
print("\n=== FINAL TEST METRICS ===")
for k, v in test_metrics.items():
    print(f"  {k}: {v}")

***** test metrics *****
  epoch                   =        3.0
  test_bleu               =     1.3082
  test_chrf++             =    17.5204
  test_loss               =     5.6477
  test_runtime            = 0:13:23.86
  test_samples_per_second =      4.354
  test_steps_per_second   =      0.137

=== FINAL TEST METRICS ===
  test_loss: 5.64772891998291
  test_bleu: 1.3082
  test_chrf++: 17.5204
  test_runtime: 803.864
  test_samples_per_second: 4.354
  test_steps_per_second: 0.137
  epoch: 3.0


# 10. save model

In [11]:
# Save best model + tokenizer (HF format: pytorch_model weights, config, vocab).
trainer.save_model(cfg.final_weights_dir)
tokenizer.save_pretrained(cfg.final_weights_dir)

# Also dump a metrics summary next to the weights.
summary = {"test_metrics": test_metrics,
           "train_metrics": train_result.metrics,
           "config": cfg.__dict__}
with open(os.path.join(cfg.final_weights_dir, "run_summary.json"), "w",
          encoding="utf-8") as f:
    json.dump(summary, f, indent=2, ensure_ascii=False)

print("[save] weights + tokenizer ->", cfg.final_weights_dir)
print("[save] files:", os.listdir(cfg.final_weights_dir))

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

[save] weights + tokenizer -> /kaggle/working/helsinki-ar-en-final
[save] files: ['config.json', 'training_args.bin', 'generation_config.json', 'run_summary.json', 'vocab.json', 'source.spm', 'tokenizer_config.json', 'target.spm', 'model.safetensors']


# 11. Sample Test

In [12]:
def translate(texts, model, tokenizer, num_beams=4):
    model.eval()
    batch = tokenizer(texts, return_tensors="pt", padding=True,
                      truncation=True, max_length=cfg.max_src_len).to(model.device)
    with torch.no_grad():
        gen = model.generate(**batch, num_beams=num_beams,
                             max_length=cfg.max_tgt_len)
    return tokenizer.batch_decode(gen, skip_special_tokens=True)

samples = raw_dsd["test"][cfg.src_col][:5]
refs = raw_dsd["test"][cfg.tgt_col][:5]
preds = translate(list(samples), trainer.model, tokenizer)
for s, r, p in zip(samples, refs, preds):
    print(f"EN : {s}\nREF: {r}\nMT : {p}\n{'-'*50}")

EN : وهذه هي الوظيفة التي أثبت ألان جرينسبان رئيس مجلس الاحتياطي الفيدرالي السابق فشله الذريع في القيام بها. ويبدو أن إخفاقه في الانتباه إلى تجاوزات السوق المالية ـ أو &quot;الخلل&quot; الصغير في تفكيره كما أشار إليه لاحقاً ـ جعله عاجزاً عن إدراك المخاطر التي جلبها عمالقة وال ستريت بابتكاراتهم المالية الجديدة. وبصفته عضواً في مجلس محافظي بنك الاحتياطي الفيدرالي تحت قيادة جرينسبان خلال الفترة 2002-2005، فإن بيرنانكي أيضاً يتحمل اللوم لأنه لم يُـبدِ أي اعتراض على سياسات جرينسبان .
REF: His blind spot on financial-market excesses – the little “flaw” in his thinking, as he later termed – left him oblivious to the dangers of Wall Street titans’ financial innovations. As a member of the Fed’s Board of Governors under Greenspan during 2002-2005, Bernanke can also be faulted for having played along.
MT : Las Focala Allan Greenspin forward picka ad hita hita hita hita hita hittash loosestash catcha catcha catcha hittash hittash hittash hittash hittash hittash hittash hitbindash hitbindash hita 

# Loading back from saved weights

In [ ]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
import torch

model_dir = "/kaggle/working/helsinki-ar-en-final"

tokenizer = AutoTokenizer.from_pretrained(model_dir)
model = AutoModelForSeq2SeqLM.from_pretrained(model_dir)

device = "cuda" if torch.cuda.is_available() else "cpu"
model = model.to(device)
model.eval()

In [ ]:
def translate(texts, model, tokenizer, num_beams=4, max_length=128):
    if isinstance(texts, str):
        texts = [texts]
    batch = tokenizer(texts, return_tensors="pt", padding=True,
                      truncation=True, max_length=max_length).to(model.device)
    with torch.no_grad():
        generated = model.generate(**batch, num_beams=num_beams,
                                   max_length=max_length)
    return tokenizer.batch_decode(generated, skip_special_tokens=True)

print(translate("Hello, how are you?", model, tokenizer))
print(translate(["The weather is nice today.", "I love reading books."],
                model, tokenizer))